# Module 03:  Fraud & Risk Scoring Engine

> **Objective:** Process high-frequency transaction logs, filter corrupted payload packets, halt execution on fraud triggers, and apply multi-tier risk evaluation rules using advanced sequence unpacking (`enumerate` + `zip`).

---

### Key Concepts Covered
* Advanced parallel unpacking: `enumerate(zip(...))`
* Stream control keywords: `continue` (skip) & `break` (halt)
* String tokenization and float type-casting
* Logical boolean evaluation (`and` / `or`)


In [5]:
# System Environment Config (Immutable Tuple)
gateway_config = ("PAY_GATEWAY_SECURE", "2026-Q3", "USD")

# Parallel Risk Tier Database
account_ids = ["ACC-8801", "ACC-8802", "ACC-8803", "ACC-8804", "ACC-8805"]
risk_tiers = ["LOW", "HIGH", "MEDIUM", "LOW", "HIGH"]

# Raw Transaction Stream Logs
raw_transaction_logs = [
    "txid:TX-101 | amount:450.00USD | status:APPROVED",
    "txid:CORRUPTED_PACKET | amount:0.00USD | status:FAILED",
    "txid:TX-103 | amount:2500.00USD | status:APPROVED",
    "txid:TX-104 | amount:12000.00USD | status:APPROVED",
    "txid:TX-105 | amount:95000.00USD | status:CRITICAL_FRAUD_TRIGGER"
]

# Step 1: Unpack immutable environment config
gateway,quarter,currency = gateway_config
gateway_info = {"Name":gateway,"Quarter":quarter,"Currency":currency}
processed_transactions = []

# Step 2: Stream ingestion loop with parallel unpacking & line indexing
for line_no,(id,risk,log) in enumerate(zip(account_ids,risk_tiers,raw_transaction_logs), start=1):
    if("CORRUPTED_PACKET" in log):          # Filter corrupted data packets
        print(f"Alert! Corrupted package at record:{line_no}")
        continue
    if("CRITICAL_FRAUD_TRIGGER" in log):        # Security halt on fraud trigger
        print(f"Alert! Fraud detected at record:{line_no}")
        break

    # Step 3: String parsing & numeric casting
    cleaned_log = log.split("|")
    txid = cleaned_log[0].split(":")[1].strip()
    amount = cleaned_log[1].split(":")[1].replace("USD","").strip()
    amount = float(amount)

    # Step 4: Multi-tier risk scoring logic
    if amount>2000.0 and risk=="HIGH":
        risk_status = "FLAGGED HIGH RISK"
    elif amount> 5000.0 or risk =="HIGH":
        risk_status = "REVIEW REQUIRED"
    else:
        risk_status = "CLEARED"
    processed_transactions.append({"ACCOUNT ID":id,"TAX ID":txid,"AMOUNT":amount,"RISK":risk_status})   # Append structured dictionary record
amount_list = [tx["AMOUNT"] for tx in processed_transactions]
# Step 5: Summary analytics & master payload assembly
summary = {"Total processed values":len(processed_transactions),"Maximium processed value":max(amount_list),"Total Volume":sum(amount_list)}
final_report = {"Gateway info":gateway_info,"Flagged records":processed_transactions,"Summary":summary}
print(final_report)

Alert! Corrupted package at record:2
Alert! Fraud detected at record:5
{'Gateway info': {'Name': 'PAY_GATEWAY_SECURE', 'Quarter': '2026-Q3', 'Currency': 'USD'}, 'Flagged records': [{'ACCOUNT ID': 'ACC-8801', 'TAX ID': 'TX-101', 'AMOUNT': 450.0, 'RISK': 'CLEARED'}, {'ACCOUNT ID': 'ACC-8803', 'TAX ID': 'TX-103', 'AMOUNT': 2500.0, 'RISK': 'CLEARED'}, {'ACCOUNT ID': 'ACC-8804', 'TAX ID': 'TX-104', 'AMOUNT': 12000.0, 'RISK': 'REVIEW REQUIRED'}], 'Summary': {'Total processed values': 3, 'Maximium processed value': 12000.0, 'Total Volume': 14950.0}}
